# Real-Channel vs. Mock: Statistical Equivalence

Tests whether your much-faster local Mock simulations are a legitimate stand-in for the
slow real-FABRIC runs. Two-proportion z-test for significance, Cohen's h for effect size
(report both — with enough trials, a trivially small difference can still show up as
"significant," so the effect size is what tells you whether it actually matters).

Run `20_data_cleaning_and_integrity.ipynb` first.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
from qne.cascade.validation_utils import two_proportion_ztest, cohens_h, mismatch_series

RESULTS = PROJECT_DIR / "results"
toeplitz_clean = pd.read_csv(str(RESULTS / "toeplitz_clean.csv"))
final_key_clean = pd.read_csv(str(RESULTS / "final_key_clean.csv"))
reconciliation_df = pd.read_csv(str(RESULTS / "reconciliation_clean.csv"))
mock_clean = pd.read_csv(str(RESULTS / "mock_clean.csv"))


In [2]:
print("=== Real (cleaned) vs Mock (cleaned): Toeplitz / Final-key ===")
for fault_name, real_clean in [("toeplitz", toeplitz_clean), ("final_key", final_key_clean)]:
    mock_sub = mock_clean[mock_clean["fault_type"] == fault_name]
    for prob in sorted(real_clean["prob"].unique()):
        real_g = real_clean[real_clean["prob"] == prob]
        mock_g = mock_sub[mock_sub["prob"] == prob]
        if len(mock_g) == 0:
            continue
        s1, n1 = real_g["mismatch"].sum(), len(real_g)
        s2, n2 = mock_g["mismatch"].sum(), len(mock_g)
        z, p = two_proportion_ztest(s1, n1, s2, n2)
        h = cohens_h(s1 / n1, s2 / n2)
        sig = "**" if p < 0.05 else ""
        print(f"  {fault_name:<12} prob={prob:<6}: real={s1}/{n1} ({s1/n1:.2f}), mock={s2}/{n2} ({s2/n2:.2f}), "
              f"z={z:.3f}, p={p:.4f} {sig}, Cohen's h={h:.3f}")


=== Real (cleaned) vs Mock (cleaned): Toeplitz / Final-key ===
  toeplitz     prob=0.001 : real=0/48 (0.00), mock=12/200 (0.06), z=-1.740, p=0.0819 , Cohen's h=-0.495
  toeplitz     prob=0.003 : real=0/48 (0.00), mock=12/200 (0.06), z=-1.740, p=0.0819 , Cohen's h=-0.495
  toeplitz     prob=0.01  : real=0/48 (0.00), mock=12/200 (0.06), z=-1.740, p=0.0819 , Cohen's h=-0.495
  toeplitz     prob=0.03  : real=0/48 (0.00), mock=12/200 (0.06), z=-1.740, p=0.0819 , Cohen's h=-0.495
  toeplitz     prob=0.05  : real=4/48 (0.08), mock=22/200 (0.11), z=-0.542, p=0.5881 , Cohen's h=-0.090
  toeplitz     prob=0.1   : real=4/48 (0.08), mock=24/200 (0.12), z=-0.721, p=0.4710 , Cohen's h=-0.122
  toeplitz     prob=0.3   : real=9/48 (0.19), mock=46/200 (0.23), z=-0.636, p=0.5245 , Cohen's h=-0.105
  toeplitz     prob=0.5   : real=12/48 (0.25), mock=58/200 (0.29), z=-0.553, p=0.5803 , Cohen's h=-0.090
  final_key    prob=0.001 : real=0/48 (0.00), mock=12/200 (0.06), z=-1.740, p=0.0819 , Cohen's h=-0.495


In [3]:
print("=== Real (cleaned) vs Mock: Reconciliation ===")
recon_mock = mock_clean[mock_clean["fault_type"] == "reconciliation"].copy()
for prob in sorted(reconciliation_df["reconciliation_prob"].unique()):
    real_g = reconciliation_df[reconciliation_df["reconciliation_prob"] == prob]
    prob_col = "reconciliation_prob" if "reconciliation_prob" in recon_mock.columns else "prob"
    mock_g = recon_mock[recon_mock[prob_col] == prob]
    if len(real_g) == 0 or len(mock_g) == 0:
        print(f"  prob={prob}: skipped -- missing data on one side")
        continue
    s1, n1 = real_g["mismatch"].sum(), len(real_g)
    s2, n2 = mock_g["mismatch"].sum(), len(mock_g)
    z, p = two_proportion_ztest(s1, n1, s2, n2)
    h = cohens_h(s1 / n1, s2 / n2)
    sig = "**" if p < 0.05 else ""
    print(f"  prob={prob:<6}: real={s1}/{n1} ({s1/n1:.2f}), mock={s2}/{n2} ({s2/n2:.2f}), "
          f"z={z:.3f}, p={p:.4f} {sig}, Cohen's h={h:.3f}")


=== Real (cleaned) vs Mock: Reconciliation ===
  prob=0.001 : real=2/50 (0.04), mock=12/200 (0.06), z=-0.550, p=0.5822 , Cohen's h=-0.092
  prob=0.003 : real=2/50 (0.04), mock=22/200 (0.11), z=-1.503, p=0.1329 , Cohen's h=-0.273
  prob=0.01  : real=12/50 (0.24), mock=54/200 (0.27), z=-0.430, p=0.6669 , Cohen's h=-0.069
  prob=0.03  : real=37/50 (0.74), mock=162/200 (0.81), z=-1.099, p=0.2719 , Cohen's h=-0.168
  prob=0.05  : real=43/50 (0.86), mock=176/200 (0.88), z=-0.384, p=0.7011 , Cohen's h=-0.060
  prob=0.1   : real=50/50 (1.00), mock=200/200 (1.00), z=nan, p=nan , Cohen's h=0.000
  prob=0.3   : real=50/50 (1.00), mock=200/200 (1.00), z=nan, p=nan , Cohen's h=0.000
  prob=0.5   : real=50/50 (1.00), mock=200/200 (1.00), z=nan, p=nan , Cohen's h=0.000


## Reading this

No significant differences (or significant-but-tiny Cohen's h) across the board means: it's
legitimate to run your exploratory sweeps in Mock and reserve real-channel runs for
confirmation, not because Mock is "close enough" informally, but because you've explicitly
tested it. If any row here is both significant *and* has a non-trivial effect size, that's
worth flagging as a real Mock/real discrepancy before relying on Mock-only results for that
specific fault type/probability.